# 01 - Build the coup event list

**Question:** do currency markets move more than normal around coups d'etat?

This notebook turns the raw Powell & Thyne coup data into a clean list of events (2000-2025) for the event study.

- **Source:** Powell & Thyne, Global Instances of Coups (version 2026-08-29)
- **Input:** `data/raw/pt_20260829.csv`
- **Output:** `data/clean/events_2000_2025.csv`
- **Coding:** in the source data, `coup` = 2 is a successful coup and `coup` = 1 is a failed attempt

In [1]:
import pandas as pd

RAW_FILE = "../data/raw/pt_20260829.csv"
CLEAN_FILE = "../data/clean/events_2000_2025.csv"

START_YEAR = 2000
END_YEAR = 2025

## Load the raw data

In [2]:
df = pd.read_csv(RAW_FILE)

print("rows, columns:", df.shape)
df.head()

rows, columns: (497, 9)


,country,ccode,ccode_gw,ccode_polity,year,month,day,coup,version
0,Cuba,40,NaN,NaN,1952,3,10,2,V2026.08.29
1,Cuba,40,NaN,NaN,1957,9,5,1,V2026.08.29
2,Haiti,41,NaN,NaN,1950,5,10,2,V2026.08.29
3,Haiti,41,NaN,NaN,1956,12,12,2,V2026.08.29
4,Haiti,41,NaN,NaN,1957,4,2,2,V2026.08.29


## Add a proper date and a readable outcome label

In [3]:
# Combine the year, month and day columns into one date.
# errors="coerce" turns any impossible date into a blank instead of crashing.
df["date"] = pd.to_datetime(df[["year", "month", "day"]], errors="coerce")

# Spell out the outcome so we don't have to remember what 1 and 2 mean.
df["outcome"] = df["coup"].map({2: "successful", 1: "failed"})

# Every row should have a valid date and a known outcome.
print("rows with an invalid date:", df["date"].isna().sum())
print("rows with an unknown outcome:", df["outcome"].isna().sum())

rows with an invalid date: 0
rows with an unknown outcome: 0


## Keep events from 2000 to 2025

The 2026 Niger attempt in the source file is excluded by this window (its outcome was still being coded).

In [4]:
events = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)].copy()
events = events.sort_values("date").reset_index(drop=True)

print("events", START_YEAR, "-", END_YEAR, ":", len(events))
print()
print(events["outcome"].value_counts())
print()
print(events["country"].value_counts().head(15))

events 2000 - 2025 : 72

outcome
failed        36
successful    36
Name: count, dtype: int64

country
Guinea-Bissau                       7
Sudan                               5
Mali                                4
Burkina Faso                        4
Ivory Coast                         3
Burundi                             3
Mauritania                          3
Guinea                              3
Madagascar                          3
Niger                               3
Fiji                                2
Central African Republic            2
Sao Tome and Principe               2
Democratic Republic of the Congo    2
Chad                                2
Name: count, dtype: int64


## Save the clean event list

In [5]:
events.to_csv(CLEAN_FILE, index=False)
print("saved", len(events), "events to", CLEAN_FILE)

saved 72 events to ../data/clean/events_2000_2025.csv


In [6]:
print(events["country"].value_counts().to_string())

country
Guinea-Bissau                       7
Sudan                               5
Mali                                4
Burkina Faso                        4
Ivory Coast                         3
Burundi                             3
Mauritania                          3
Guinea                              3
Madagascar                          3
Niger                               3
Fiji                                2
Central African Republic            2
Sao Tome and Principe               2
Democratic Republic of the Congo    2
Chad                                2
Thailand                            2
Egypt                               2
Gabon                               2
Ecuador                             1
Comoros                             1
Paraguay                            1
Peru                                1
Djibouti                            1
Venezuela                           1
Togo                                1
Honduras                            1
Papu

In [7]:
uemoa = ["Benin", "Burkina Faso", "Guinea-Bissau", "Ivory Coast", "Mali", "Niger", "Senegal", "Togo"]
cemac = ["Cameroon", "Central African Republic", "Chad", "Congo", "Equatorial Guinea", "Gabon"]
euro_peg = ["Sao Tome and Principe", "Comoros"]

events["excluded_reason"] = None
events.loc[events["country"].isin(uemoa), "excluded_reason"] = "West African CFA franc (euro peg)"
events.loc[events["country"].isin(cemac), "excluded_reason"] = "Central African CFA franc (euro peg)"
events.loc[events["country"].isin(euro_peg), "excluded_reason"] = "Direct euro peg"

print(events["excluded_reason"].value_counts(dropna=False))


excluded_reason
None                                    40
West African CFA franc (euro peg)       23
Central African CFA franc (euro peg)     6
Direct euro peg                          3
Name: count, dtype: int64


In [8]:
remaining = sorted(events.loc[events["excluded_reason"].isna(), "country"].unique())
print(len(remaining), "countries remain:")
print(remaining)


24 countries remain:
['Bangladesh', 'Bolivia', 'Burundi', 'Democratic Republic of the Congo', 'Djibouti', 'Ecuador', 'Egypt', 'Fiji', 'Guinea', 'Honduras', 'Lesotho', 'Madagascar', 'Maldives', 'Mauritania', 'Myanmar', 'Papua New Guinea', 'Paraguay', 'Peru', 'Sudan', 'Thailand', 'Turkey', 'Ukraine', 'Venezuela', 'Zimbabwe']


In [9]:
events.to_csv("../data/clean/events_2000_2025_tagged.csv", index=False)
print("saved", len(events), "rows;", events["excluded_reason"].isna().sum(), "remain after the peg filter")


saved 72 rows; 40 remain after the peg filter
